# MART_P04_OCUPACAO_HOSPITALAR

**P04. Quais hospitais apresentam a maior taxa de ocupação em relação ao volume de
leitos?** *(dashboard)*

- **Pacientes-dia:** soma dos dias de permanência das internações com saída no mês (SIH).
- **Leitos-dia:** leitos SUS do estabelecimento no mês (CNES) × número de dias do mês.
- **Taxa de ocupação** = pacientes-dia ÷ leitos-dia × 100.

É uma aproximação: a internação inteira é atribuída ao mês da saída. Valores acima de
100% indicam capacidade esgotada no mês (ou leitos não atualizados no CNES).

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO MART_P04_OCUPACAO_HOSPITALAR
    (ANO_MES, COD_CNES, NOME_ESTABELECIMENTO, NOME_MUNICIPIO, QTD_LEITOS_SUS,
     QTD_PACIENTES_DIA, QTD_LEITOS_DIA, PCT_OCUPACAO)
WITH pacientes AS (
    SELECT t.ANO_MES, f.SK_ESTABELECIMENTO, SUM(f.QTD_DIAS_PERMANENCIA) AS PACIENTES_DIA
    FROM FATO_INTERNACAO f
    JOIN DIM_TEMPO t ON t.SK_TEMPO = f.SK_TEMPO_SAIDA
    GROUP BY t.ANO_MES, f.SK_ESTABELECIMENTO
),
leitos AS (
    SELECT t.ANO_MES, l.SK_ESTABELECIMENTO, l.QTD_LEITOS_SUS,
           l.QTD_LEITOS_SUS * EXTRACT(DAY FROM LAST_DAY(t.DT_COMPLETA)) AS LEITOS_DIA
    FROM FATO_LEITOS l
    JOIN DIM_TEMPO t ON t.SK_TEMPO = l.SK_TEMPO_COMPETENCIA
    WHERE l.QTD_LEITOS_SUS > 0
)
SELECT l.ANO_MES, e.COD_CNES, e.NOME_ESTABELECIMENTO, e.NOME_MUNICIPIO, l.QTD_LEITOS_SUS,
       NVL(p.PACIENTES_DIA, 0), l.LEITOS_DIA,
       ROUND(100 * NVL(p.PACIENTES_DIA, 0) / l.LEITOS_DIA, 2)
FROM leitos l
JOIN DIM_ESTABELECIMENTO e ON e.SK_ESTABELECIMENTO = l.SK_ESTABELECIMENTO
LEFT JOIN pacientes p      ON p.ANO_MES = l.ANO_MES AND p.SK_ESTABELECIMENTO = l.SK_ESTABELECIMENTO
WHERE p.PACIENTES_DIA IS NOT NULL
"""

In [ ]:
recarregar("MART_P04_OCUPACAO_HOSPITALAR", SQL, origem="FATO_LEITOS")

In [ ]:
consultar("SELECT * FROM MART_P04_OCUPACAO_HOSPITALAR FETCH FIRST 10 ROWS ONLY")